# Dastan — how it works, end to end

An open-source expected-points model for Fantasy Premier League.

This notebook is the guided version of the repository. It builds the model from the
frame, explains the two or three decisions that actually matter, and reproduces the
published numbers.

**If you read only one section, read §3.** Leakage is the reason most FPL models report
numbers they cannot reproduce, and the specific way it happens here is subtle.

## 1. Setup

Everything is local — no API keys, no scraping, no database. The frame and the weights
are both in the repository.

In [1]:
import sys, json
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from dastan import data, metrics, model, predictor

pd.set_option("display.width", 120)
print("ready")

ready


## 2. The data

One row per player per fixture, six seasons. `data.load()` joins the pre-deadline
snapshot artefacts and fills every missing value with `-1`.

In [2]:
df = data.load()
feats = data.shipped_features(df)

print(f"{len(df):,} rows, {len(feats)} features")
print(df.groupby("season").agg(rows=("gameweek", "size"), gameweeks=("gameweek", "nunique")))

163,072 rows, 286 features
          rows  gameweeks
season                   
2020-21  24365         38
2021-22  25447         38
2022-23  26505         37
2023-24  29725         38
2024-25  27283         38
2025-26  29747         38


### Why missing is `-1` and not `0`

54,430 rows carry a genuine `ep_next` of exactly zero — FPL expecting nothing from a
player. If absent snapshots were filled with `0`, those two very different events would
be identical to the model.

In [3]:
ep = df["ar_ep_next"]
print(f"rows with no snapshot (-1)      : {(ep == -1).sum():,}")
print(f"rows where FPL genuinely said 0 : {(ep == 0).sum():,}")
print(f"mean where present              : {ep[ep != -1].mean():.3f}")

rows with no snapshot (-1)      : 23,387
rows where FPL genuinely said 0 : 54,430
mean where present              : 1.236


## 3. Leakage — the part that matters

Two properties have to hold. Both are easy to get wrong and neither raises an error when
you do.

### 3.1 Targets are not in the features

Five columns in the frame describe the gameweek being predicted. They are in the frame
so rolling versions can be rebuilt — **using them directly is leakage**. Watch how
cleanly they give the answer away:

In [4]:
for c in ["starts", "defensive_contribution", "cbit", "recoveries", "tackles"]:
    s, m = df[c], df["minutes"].fillna(0)
    ok = s.notna()
    zero_when_blank = (s[ok & (m == 0)] == 0).mean()
    print(f"{c:24} corr(minutes) {np.corrcoef(s[ok], m[ok])[0,1]:+.3f}   "
          f"zero when player didn't play: {zero_when_blank:.1%}")

print(f"\nin the feature set? {[c for c in ['starts','cbit','tackles'] if c in feats]}")

starts                   corr(minutes) +0.902   zero when player didn't play: 100.0%
defensive_contribution   corr(minutes) +0.783   zero when player didn't play: 100.0%
cbit                     corr(minutes) +0.667   zero when player didn't play: 100.0%
recoveries               corr(minutes) +0.788   zero when player didn't play: 100.0%
tackles                  corr(minutes) +0.604   zero when player didn't play: 100.0%

in the feature set? []


### 3.2 The deadline is not the kickoff

This is the subtle one.

Rolling features are built by shifting one row back, ordered by kickoff. That correctly
keeps a fixture out of its own feature vector. But **FPL squads lock at a deadline, and
a double gameweek has two fixtures behind one deadline.** Sorted by kickoff, the second
fixture's "previous match" is the first fixture of the same gameweek — a result nobody
had seen when the squad was picked.

The frame shipped here is already corrected. The check runs on every load:

In [5]:
dupes = df[df.duplicated(["season", "gameweek", "fpl_code"], keep=False)]
print(f"player-gameweeks with more than one fixture: "
      f"{dupes.groupby(['season','gameweek','fpl_code']).ngroups:,}")

data.assert_deadline_anchored(df)      # raises if any history column varies within one
print("deadline-anchored: every fixture in a gameweek shares one history snapshot")

player-gameweeks with more than one fixture: 6,958
deadline-anchored: every fixture in a gameweek shares one history snapshot


### 3.3 Provenance for anything from a snapshot

`ep_next` and the availability flags come from an archive of the FPL bootstrap endpoint.
A snapshot is accepted **only** if it names the gameweek as `is_next`, agrees on its
deadline, and was captured before it.

Without that test these columns are the answer key. Concretely: an unchecked version of
`ep_next` scores NDCG@10 of **0.4062**; the provenance-checked one scores **0.2939**.
The difference is entirely post-deadline captures.

## 4. The architecture

About 62% of player-gameweeks are zero minutes, and among players who start, points are
violently right-skewed. One regressor across that mixture learns mostly that people
score nothing, and never predicts a haul.

So the target is decomposed:

$$\text{xPts} = p_{60}\sum_k P(\text{band }k \mid \text{started})\,
E[\text{pts} \mid \text{band }k]  +  (1-p_{60})\,E[\text{pts} \mid \text{didn't start}]$$

In [6]:
print(f"zero-minute share: {(df['minutes'].fillna(0) == 0).mean():.1%}")

started = df[df["minutes"].fillna(0) >= 60]
print("\npoints among players who started 60+:")
print(started["target_points"].describe(percentiles=[.5, .9, .99]).round(2))
print(f"\nband edges {model.BUCKET_EDGES} -> "
      f"{np.bincount(np.digitize(started['target_points'], model.BUCKET_EDGES), minlength=4)}")

zero-minute share: 59.1%

points among players who started 60+:
count    47291.00
mean         3.54
std          3.18
min         -7.00
50%          2.00
90%          8.00
99%         15.00
max         26.00
Name: target_points, dtype: float64

band edges [1, 3, 10] -> [ 2320 25253 17020  2698]


**Each head early-stops on the population it models.** The "given under 60 minutes"
head early-stops on under-60 rows, not on a validation set dominated by starters.

This sounds pedantic. It is not — fixing this class of defect was worth more to this
model than any feature in it.

## 5. Train one position

Chronological split: train through 2025-26 GW30, hold out GW31-38 for early stopping,
calibration and the blend weight.

In [7]:
ts = df["season"].eq("2025-26")
hold = ts & df["gameweek"].between(31, 38)
pool = df[~(ts & df["gameweek"].gt(30))]
train, val = pool[~hold.reindex(pool.index, fill_value=False)], df[hold]
print(f"train {len(train):,}  holdout {len(val):,}")

models, calibration, blend_w = model.fit_position(train, val, feats, "MID", n_jobs=8)
print(f"\nMID blend weight {blend_w}  calibration {calibration}")

train 156,490  holdout 6,582



MID blend weight 0.45  calibration {'gamma': 1.0, 'w0': 1.0, 'w1': 1.0, 'w2': 1.0, 'w3': 1.0}


The blend weight is fitted **by the ranking objective**, not by RMSE. Optimising a
metric the model is not judged on picks a different weight.

## 6. Scoring — inside a gameweek, two cohorts

Metrics are computed within each gameweek and then averaged. Pooling every
player-gameweek in a season flatters a model badly, because most of the variance is then
between gameweeks — a blank week against a double — rather than the question a manager
faces: *given this week, who do I pick?*

In [8]:
v = val[val["position"].eq("MID")]
pred = model.predict(models, calibration, blend_w, v[feats].fillna(0.0).to_numpy())

pf = metrics.collapse_to_player_gameweek(
    v.assign(pred=pred, actual=v["target_points"]), "pred")

for cohort in ("all", "starters"):
    s = metrics.score(pf, "pred", cohort)
    print(f"{cohort:9} obj {s['obj']:.4f}  spearman {s['spearman']:.4f}  "
          f"ndcg@10 {s['ndcg@10']:.4f}  mae {s['mae']:.3f}  n={s['rows']:,}")

all       obj 0.6203  spearman 0.7711  ndcg@10 0.4696  mae 0.845  n=2,806
starters  obj 0.3229  spearman 0.1476  ndcg@10 0.4982  mae 2.226  n=641


The two cohorts answer different questions, and **they disagree**:

- `all` — ~62% never played, so this rewards knowing *who will not play*
- `starters` — 60+ minutes, much harder, and closer to the real decision

`starters` is always reported and **never optimised against**, so a model that wins only
by predicting non-players is visible rather than hidden.

## 7. The noise floor

Before believing any result, measure how much the objective moves when nothing changes
but the seed.

On this data the seed-to-seed range is **0.0106** — larger than almost every feature
effect anyone will report on FPL data. With three seeds the keep margin is
`0.0106 / sqrt(3) = 0.0061`.

An earlier version of this project used a margin of 0.0020: **5.3× below the noise it
was screening against.** A protocol like that does not detect signal, it manufactures
it. One family entered the model on a "+0.0058" that was entirely inside its own error
bar, and is still there — documented in `docs/FEATURES.md` rather than quietly justified.

Run this if you want to see it for yourself (a few minutes per seed):

In [9]:
# for seed in (42, 7, 2026):
#     m, c, w = model.fit_position(train, val, feats, "MID", n_jobs=8, seed=seed)
#     p = model.predict(m, c, w, v[feats].fillna(0.0).to_numpy())
#     f = metrics.collapse_to_player_gameweek(v.assign(pred=p, actual=v["target_points"]), "pred")
#     print(seed, round(metrics.score(f, "pred", "all")["obj"], 4))
print("uncomment to measure your own noise floor before trusting any feature result")

uncomment to measure your own noise floor before trusting any feature result


## 8. The released weights

`predictor.Dastan` reloads all 37 artefacts from disk the way an application would.

In [10]:
m = predictor.Dastan()
test = df[df["season"].eq("2025-26") & df["gameweek"].between(31, 38)]
out = m.predict_frame(test, with_parts=True)

# Names live in a separate lookup: the frame is keyed by fpl_code, which is stable
# across seasons. FPL's `element` id is not -- 1,130 of 1,959 players in this dataset
# had theirs change between seasons, so joining on it silently merges careers.
players = pd.read_csv("../data/players.csv")
out = out.merge(players[["season", "fpl_code", "player_name"]],
                on=["season", "fpl_code"], how="left")

print(out.nlargest(10, "xpts")[
    ["player_name", "team_name", "position", "gameweek", "xpts", "p60"]
].round(3).to_string(index=False))

                 player_name     team_name position  gameweek  xpts   p60
             Antoine Semenyo      Man City      MID        33 6.014 0.920
      Bruno Borges Fernandes       Man Utd      MID        32 5.568 0.969
                 Bukayo Saka       Arsenal      MID        37 5.466 0.761
      Bruno Borges Fernandes       Man Utd      MID        31 5.270 0.968
          Morgan Gibbs-White Nott'm Forest      MID        33 5.226 0.926
                Harry Wilson        Fulham      MID        31 5.173 0.899
      Bruno Borges Fernandes       Man Utd      MID        35 5.169 0.977
      Bruno Borges Fernandes       Man Utd      MID        38 4.996 0.967
              Erling Haaland      Man City      FWD        33 4.961 0.865
Gabriel dos Santos Magalhães       Arsenal      DEF        37 4.944 0.931


### The band probabilities say something `xpts` cannot

Two players can share a projection and mean completely different things — a steady
five, or a coin-flip between two and thirteen.

In [11]:
bands = ["p_band0", "p_band1", "p_band2", "p_band3"]
top = out.nlargest(6, "xpts")
print(top[["player_name", "xpts", "p60"] + bands].round(3).to_string(index=False))
print("\nbands: 0 = under 1 pt, 1 = 1-3, 2 = 3-10, 3 = 10+  (given the player starts)")

           player_name  xpts   p60  p_band0  p_band1  p_band2  p_band3
       Antoine Semenyo 6.014 0.920    0.006    0.252    0.548    0.195
Bruno Borges Fernandes 5.568 0.969    0.005    0.304    0.506    0.185
           Bukayo Saka 5.466 0.761    0.003    0.183    0.587    0.227
Bruno Borges Fernandes 5.270 0.968    0.005    0.366    0.447    0.182
    Morgan Gibbs-White 5.226 0.926    0.005    0.390    0.452    0.152
          Harry Wilson 5.173 0.899    0.004    0.301    0.530    0.166

bands: 0 = under 1 pt, 1 = 1-3, 2 = 3-10, 3 = 10+  (given the player starts)


## 9. Where this model is weak

Stated plainly, because it changes what you should build on top of it.

1. **The added features do nothing for starters.** Against a core feature set, Dastan
   gains +0.0128 on all players and **+0.0010 on starters** — below the noise floor. The
   OpenFPL head-to-head reaches the same conclusion independently (+0.0558 all, +0.0025
   starters). Everything the feature work buys is in predicting participation.

2. **Absolute values are conservative for starters** — under-predicted by 0.83 points on
   average. It ranks well; do not read the level as a point estimate.

3. **Premier League fixtures only.** A player who went 90 minutes in the Europa League on
   Thursday looks fully rested. This is the most promising open lead in the repo.

4. **Top-10 overlap is 0.183** — about 1.8 of a gameweek's ten highest scorers appear in
   our predicted ten. That is near the realistic ceiling; gameweek tops are dominated by
   hauls that are close to irreducibly random.

`docs/FEATURES.md` lists eleven candidate families that were built, measured and thrown
away, with numbers — including several that sound obviously good.